# 🎵 Amapiano-AI Complete Training Pipeline

This notebook provides a complete end-to-end training pipeline.

**Run cells in order from top to bottom.**

**Sections:**
1. Install Dependencies
2. Configuration
3. Download Dataset
4. Dataset Class
5. Data Augmentation
6. CNN Model
7. Training Functions
8. SVDQuant Quantization
9. Run Pipeline

---
## Section 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchaudio librosa pandas numpy scikit-learn matplotlib tqdm
!pip install -q onnx onnxruntime onnxscript

# Install audio codec support
!apt-get -qq install -y ffmpeg libsndfile1

print("✅ Dependencies installed")

---
## Section 2: Configuration

In [ ]:
import os
import torch
import numpy as np
import random

# Set random seeds
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Configuration
CONFIG = {
    'data_dir': '/content/magnatagatune',
    'output_dir': '/content/output',
    'sample_rate': 22050,
    'duration': 5,
    'n_mels': 128,
    'n_fft': 2048,
    'hop_length': 512,
    'batch_size': 32,
    'num_epochs': 10,
    'learning_rate': 0.001,
    'weight_decay': 1e-4,
    'hidden_dim': 256,
    'dropout': 0.3,
    'target_tags': ['drums', 'bass', 'synth', 'electronic', 'dance', 'beat', 'percussion', 'deep', 'house']
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✅ CONFIG defined")
print(f"📱 Device: {device}")
print(f"🎵 Sample rate: {CONFIG['sample_rate']} Hz")
print(f"🏷️ Tags: {CONFIG['target_tags']}")

---
## Section 3: Download Dataset

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

data_dir = Path(CONFIG['data_dir'])
data_dir.mkdir(parents=True, exist_ok=True)

annotations_path = data_dir / 'annotations_final.csv'
mp3_zip_path = data_dir / 'mp3.zip'

# Download annotations
if not annotations_path.exists():
    print("📥 Downloading annotations...")
    urllib.request.urlretrieve(
        'https://mirg.city.ac.uk/datasets/magnatagatune/annotations_final.csv',
        str(annotations_path)
    )
    print("✅ Annotations downloaded")
else:
    print("✅ Annotations exist")

# Download MP3s
if not mp3_zip_path.exists() and not (data_dir / 'mp3').exists():
    print("📥 Downloading MP3 files (~1.5GB, may take a while)...")
    urllib.request.urlretrieve(
        'https://huggingface.co/datasets/confit/magnatagatune/resolve/main/mp3.zip',
        str(mp3_zip_path)
    )
    print("✅ MP3 zip downloaded")
    
    print("📦 Extracting MP3 files...")
    with zipfile.ZipFile(str(mp3_zip_path), 'r') as z:
        z.extractall(str(data_dir))
    print("✅ Extracted")
else:
    print("✅ MP3 files exist")

mp3_files = list(data_dir.rglob('*.mp3'))
print(f"🎵 Total MP3 files: {len(mp3_files)}")

---
## Section 4: Dataset Class

In [ ]:
import pandas as pd
import librosa
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

class MagnaTagATuneDataset(Dataset):
    def __init__(self, config, split='train', transform=None):
        self.config = config
        self.split = split
        self.transform = transform
        self.sample_rate = config['sample_rate']
        self.duration = config['duration']
        self.n_samples = self.sample_rate * self.duration
        
        # Load annotations
        annotations_path = Path(config['data_dir']) / 'annotations_final.csv'
        print(f"Loading annotations from {annotations_path}")
        self.annotations = pd.read_csv(annotations_path, sep='\t')
        
        # Get tag columns
        non_tag_cols = ['mp3_path', 'clip_id']
        all_tag_cols = [col for col in self.annotations.columns if col not in non_tag_cols]
        
        if config.get('target_tags'):
            self.tag_columns = [t for t in config['target_tags'] if t in all_tag_cols]
        else:
            self.tag_columns = all_tag_cols[:50]
        
        print(f"Using {len(self.tag_columns)} tags: {self.tag_columns}")
        
        # Split data
        n_total = len(self.annotations)
        indices = np.random.permutation(n_total)
        train_end = int(0.8 * n_total)
        val_end = int(0.9 * n_total)
        
        if split == 'train':
            self.indices = indices[:train_end]
        elif split == 'val':
            self.indices = indices[train_end:val_end]
        else:
            self.indices = indices[val_end:]
        
        print(f"{split} split: {len(self.indices)} samples")
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        row = self.annotations.iloc[real_idx]
        mp3_path = Path(self.config['data_dir']) / row['mp3_path']
        
        try:
            audio, sr = librosa.load(str(mp3_path), sr=self.sample_rate, duration=self.duration)
            if len(audio) < self.n_samples:
                audio = np.pad(audio, (0, self.n_samples - len(audio)))
            else:
                audio = audio[:self.n_samples]
            if self.transform:
                audio = self.transform(audio)
        except:
            audio = np.zeros(self.n_samples)
        
        labels = row[self.tag_columns].values.astype(np.float32)
        return torch.tensor(audio, dtype=torch.float32), torch.tensor(labels, dtype=torch.float32)

print("✅ Dataset class defined")

---
## Section 5: Data Augmentation

In [ ]:
class AudioAugmentation:
    def __init__(self, sample_rate=22050):
        self.sample_rate = sample_rate
    
    def __call__(self, audio):
        if np.random.random() < 0.5:
            audio = audio + np.random.randn(len(audio)) * 0.005
        if np.random.random() < 0.3:
            shift = int(len(audio) * np.random.uniform(-0.2, 0.2))
            audio = np.roll(audio, shift)
        if np.random.random() < 0.3:
            audio = audio * np.random.uniform(0.8, 1.2)
        return audio

augmentation = AudioAugmentation(CONFIG['sample_rate'])
print("✅ Augmentation defined")

---
## Section 6: CNN Model

In [ ]:
import torch.nn as nn
import torchaudio.transforms as T

class AudioCNN(nn.Module):
    def __init__(self, num_classes, config):
        super().__init__()
        self.mel_spec = T.MelSpectrogram(
            sample_rate=config['sample_rate'],
            n_fft=config['n_fft'],
            hop_length=config['hop_length'],
            n_mels=config['n_mels']
        )
        self.amplitude_to_db = T.AmplitudeToDB()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.3),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d((4, 4)), nn.Dropout(0.3)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, config['hidden_dim']),
            nn.ReLU(),
            nn.Dropout(config['dropout']),
            nn.Linear(config['hidden_dim'], num_classes),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        x = self.mel_spec(x)
        x = self.amplitude_to_db(x)
        x = x.unsqueeze(1)
        x = self.conv_layers(x)
        return self.classifier(x)

print("✅ CNN model defined")

---
## Section 7: Training Functions

In [ ]:
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
import torch.optim as optim

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for audio, labels in tqdm(loader, desc="Training", leave=False):
        audio, labels = audio.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(audio), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for audio, labels in tqdm(loader, desc="Validating", leave=False):
            audio, labels = audio.to(device), labels.to(device)
            outputs = model(audio)
            total_loss += criterion(outputs, labels).item()
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    try:
        auc_scores = [roc_auc_score(all_labels[:, i], all_preds[:, i]) 
                      for i in range(all_labels.shape[1]) if len(np.unique(all_labels[:, i])) > 1]
        avg_auc = np.mean(auc_scores) if auc_scores else 0.5
    except:
        avg_auc = 0.5
    return total_loss / len(loader), avg_auc

def train_model(model, train_loader, val_loader, config, device):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    
    best_auc = 0
    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    
    for epoch in range(config['num_epochs']):
        print(f"\nEpoch {epoch+1}/{config['num_epochs']}")
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_auc = validate(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(val_auc)
        
        print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f}")
        
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), f"{config['output_dir']}/best_model.pt")
            print(f"  ✅ Best model saved (AUC: {best_auc:.4f})")
    
    return history

print("✅ Training functions defined")

---
## Section 8: SVDQuant Quantization

In [ ]:
class SVDQuantAudio:
    def __init__(self, bit_depth=8):
        self.bit_depth = bit_depth
        self.levels = 2 ** bit_depth
        self.target_fad = {4: 0.25, 8: 0.15, 16: 0.05}.get(bit_depth, 0.15)
        self.transient_threshold = {4: 0.3, 8: 0.5, 16: 0.7}.get(bit_depth, 0.5)
    
    def quantize(self, audio):
        if isinstance(audio, torch.Tensor):
            audio = audio.numpy()
        transients = self._detect_transients(audio)
        quantized = np.zeros_like(audio)
        for i, (sample, is_transient) in enumerate(zip(audio, transients)):
            bits = min(self.bit_depth + 4, 16) if is_transient else self.bit_depth
            normalized = (sample + 1) / 2
            levels = 2 ** bits
            quantized[i] = np.round(normalized * (levels - 1)) / (levels - 1) * 2 - 1
        return torch.tensor(self._apply_dither(quantized), dtype=torch.float32)
    
    def _detect_transients(self, audio):
        frame_size, hop_size = 512, 256
        n_frames = (len(audio) - frame_size) // hop_size + 1
        energies = np.array([np.sum(audio[i*hop_size:i*hop_size+frame_size]**2) for i in range(n_frames)])
        energy_diff = np.abs(np.diff(energies, prepend=energies[0]))
        threshold = np.mean(energy_diff) + self.transient_threshold * np.std(energy_diff)
        transient_frames = energy_diff > threshold
        transients = np.zeros(len(audio), dtype=bool)
        for i, is_t in enumerate(transient_frames):
            if is_t:
                transients[i*hop_size:min(i*hop_size+frame_size, len(audio))] = True
        return transients
    
    def _apply_dither(self, audio, amount=1e-5):
        return np.clip(audio + np.random.uniform(-amount, amount, len(audio)) * 2, -1, 1)
    
    def calculate_metrics(self, original, quantized):
        if isinstance(original, torch.Tensor): original = original.numpy()
        if isinstance(quantized, torch.Tensor): quantized = quantized.numpy()
        mse = np.mean((original - quantized) ** 2)
        snr = 10 * np.log10(np.mean(original ** 2) / (mse + 1e-10))
        fad = np.mean(np.abs(np.abs(np.fft.fft(original)) - np.abs(np.fft.fft(quantized)))) / (np.mean(np.abs(np.fft.fft(original))) + 1e-10)
        return {'snr_db': float(snr), 'fad': float(fad), 'compression': 32/self.bit_depth, 'passed': fad < self.target_fad}

print("✅ SVDQuant defined")

---
## Section 9: Run Complete Pipeline

In [ ]:
print("="*60)
print("STEP 1: Creating datasets")
print("="*60)

train_dataset = MagnaTagATuneDataset(CONFIG, split='train', transform=augmentation)
val_dataset = MagnaTagATuneDataset(CONFIG, split='val')

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

num_classes = len(train_dataset.tag_columns)
print(f"Classes: {num_classes}, Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
print("="*60)
print("STEP 2: Verify data loading")
print("="*60)

sample_audio, sample_labels = next(iter(train_loader))
print(f"Audio: {sample_audio.shape}, Labels: {sample_labels.shape}")
print(f"Audio range: [{sample_audio.min():.4f}, {sample_audio.max():.4f}]")

In [ ]:
print("="*60)
print("STEP 3: Create model")
print("="*60)

model = AudioCNN(num_classes, CONFIG).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")

with torch.no_grad():
    test_out = model(sample_audio.to(device))
print(f"Output: {test_out.shape}")

In [ ]:
print("="*60)
print("STEP 4: Train model")
print("="*60)

history = train_model(model, train_loader, val_loader, CONFIG, device)
print(f"\n✅ Training complete! Best AUC: {max(history['val_auc']):.4f}")

In [ ]:
print("="*60)
print("STEP 5: Quantization test")
print("="*60)

test_audio = sample_audio[0].numpy()

for bits in [8, 4]:
    quantizer = SVDQuantAudio(bit_depth=bits)
    quantized = quantizer.quantize(test_audio)
    metrics = quantizer.calculate_metrics(test_audio, quantized)
    print(f"\n{bits}-bit: SNR={metrics['snr_db']:.1f}dB, FAD={metrics['fad']:.4f}, {'✅' if metrics['passed'] else '❌'}")

In [ ]:
print("="*60)
print("STEP 6: Export to ONNX")
print("="*60)

!pip install -q onnxscript

model.eval()
dummy = torch.randn(1, CONFIG['sample_rate'] * CONFIG['duration']).to(device)

try:
    onnx_path = f"{CONFIG['output_dir']}/audio_cnn.onnx"
    torch.onnx.export(model, dummy, onnx_path, export_params=True, opset_version=14,
                      input_names=['audio'], output_names=['predictions'],
                      dynamic_axes={'audio': {0: 'batch'}, 'predictions': {0: 'batch'}})
    print(f"✅ Exported to {onnx_path}")
except Exception as e:
    print(f"⚠️ ONNX export failed: {e}")

In [ ]:
print("="*60)
print("STEP 7: Plot results")
print("="*60)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history['val_auc'], 'g-')
axes[1].set_title('Validation AUC'); axes[1].grid(True)

plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/training_history.png", dpi=150)
plt.show()

print(f"\n✅ PIPELINE COMPLETE!")
print(f"Best AUC: {max(history['val_auc']):.4f}")
print(f"Model: {CONFIG['output_dir']}/best_model.pt")